# 따릉이 스테이션 별 수요도 예측 베이스라인

이 노트북은 DB 연동부터 데이터 전처리, 여러 머신러닝 모델의 비교 및 하이퍼파라미터 튜닝, 앙상블 기법을 적용하여 최종 수요 예측 모델을 완성하는 파이프라인입니다.

## 환경 설정 (필수 라이브러리 로드)

In [ ]:
# 필요 시 아래 주석을 풀고 라이브러리를 설치하세요.
# !pip install pandas scikit-learn joblib matplotlib seaborn sqlalchemy pymysql python-dotenv xgboost lightgbm optuna

import sys
import os
sys.path.append(os.path.dirname(os.getcwd())) # 상위 폴더(프로젝트 루트)를 모듈 검색 경로에 추가.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

plt.rcParams["axes.unicode_minus"] = False # 그래프에서 음수 부호(-)가 깨지는 문제 방지
# 한글 폰트 설정 (Windows: Malgun Gothic, Mac: AppleGothic)
plt.rcParams['font.family'] = 'Malgun Gothic'

SEED = 42 # 난수 시드 고정
np.random.seed(SEED)

print("라이브러리 로드 완료")

## 데이터베이스 연결 및 데이터 조회
데이터베이스에 연결하여 예측에 필요한 데이터를 DataFrame(`df`)으로 불러옵니다.

In [ ]:
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "3306")
DB_NAME     = os.getenv("DB_NAME", "seoul_bike") # 첨부 이미지 참조

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    print("데이터베이스 연결 성공" if conn.execute(text("SELECT 1")).scalar() == 1 else "데이터베이스 연결 실패")

# ---------------------------------------------------------
# [주의] 아래는 df가 생성되었다고 가정하는 구간입니다.
# 쿼리를 통해 실제로 df를 불러오는 코드를 이곳에 작성하세요.
# 예시: df = pd.read_sql("SELECT * FROM rt_bike_status ...", con=engine)
# ---------------------------------------------------------

## 전처리 및 EDA (탐색적 데이터 분석)
결측치를 처리하고 데이터의 전반적인 분포를 시각화하여 확인합니다.

In [ ]:
# df가 로드되어 있다고 가정하고 진행합니다.
# (실행을 위해 DB 로드 코드가 완성되어 있어야 합니다)

print("데이터 정보:")
df.info()

print("\n결측치 확인:")
print(df.isnull().sum())

# 타겟 변수(occupancy_pct 혹은 해당 수요량 컬럼) 분포 확인
plt.figure(figsize=(10, 5))
sns.histplot(df['occupancy_pct'], bins=50, kde=True)
plt.title("타겟 변수(수요도/혼잡도) 분포")
plt.xlabel("Occupancy Percentage")
plt.ylabel("빈도")
plt.show()

# 피처 간 상관관계 확인 (수치형 데이터만)
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("피처 간 상관관계 히트맵")
plt.show()

## 피처(X) / 타깃(Y) 분리
시계열성을 유지하기 위해 날짜 기준으로 정렬한 뒤 분리합니다.

In [ ]:
FEATURE_COLUMNS = [
    "lot_id", "capacity", "month", "day_of_week",
    "is_weekend", "week_of_year", "is_holiday",
]
TARGET_COLUMN = "occupancy_pct"

# 시간 순서 유지 (미래 정보 누수 방지)
df_sorted = df.sort_values("use_date").reset_index(drop=True)

X = df_sorted[FEATURE_COLUMNS]
y = df_sorted[TARGET_COLUMN]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
display(X.head())

## Train / Validation / Test 3분할
전체 데이터를 60% / 20% / 20% 비율로 순서대로 분할합니다.

In [ ]:
n = len(X)
train_end = int(n * 0.60)
val_end   = int(n * 0.80)

X_train, y_train = X.iloc[:train_end],        y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],          y.iloc[val_end:]

print(f"Train : {len(X_train):,}행")
print(f"Val   : {len(X_val):,}행")
print(f"Test  : {len(X_test):,}행")

## 평가 지표 함수 및 기본 모델 학습 (Baseline)
RMSLE, RMSE, MAE를 계산하는 함수를 만들고, LinearRegression을 기준으로 설정합니다.

In [ ]:
def rmsle(y, pred):
    log_y    = np.log1p(y)
    log_pred = np.log1p(np.maximum(pred, 0))
    return np.sqrt(np.mean((log_y - log_pred) ** 2))

def rmse(y, pred):
    return np.sqrt(mean_squared_error(y, pred))

def evaluate_regr(y, pred, name=""):
    rmsle_val = rmsle(y, pred)
    rmse_val  = rmse(y, pred)
    mae_val   = mean_absolute_error(y, pred)
    label = f"[{name}] " if name else ""
    print(f"{label}RMSLE: {rmsle_val:.4f}, RMSE: {rmse_val:.3f}, MAE: {mae_val:.3f}")
    return {"name": name, "rmsle": rmsle_val, "rmse": rmse_val, "mae": mae_val}

# 8. 기본 모델 (Baseline)
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

val_pred = baseline_model.predict(X_val)
baseline_result = evaluate_regr(y_val, val_pred, name="LinearRegression(baseline)")

## 여러 모델 비교 (Ridge, RandomForest, XGBoost, LightGBM)
베이스라인보다 나은 성능을 보여주는 알고리즘을 찾기 위해 다양한 모델을 Validation 데이터로 평가합니다.

In [ ]:
# 비교할 모델들 정의
models = {
    "Ridge": Ridge(alpha=1.0, random_state=SEED),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=SEED, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=SEED, n_jobs=-1, verbose=-1)
}

results_list = [baseline_result]

print("[Validation 셋 기준 모델 비교]")
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    res = evaluate_regr(y_val, pred, name=name)
    results_list.append(res)

# 결과를 보기 쉽게 DataFrame으로 정리
comparison_df = pd.DataFrame(results_list).sort_values("rmsle").reset_index(drop=True)
display(comparison_df)

## 하이퍼파라미터 튜닝 (Optuna 적용)
일반적으로 트리 기반 모델(LightGBM)이 성능이 우수합니다. Optuna를 활용해 LightGBM의 파라미터를 튜닝합니다.
*(시간 단축을 위해 n_trials=20으로 설정했습니다. 성능을 더 올리려면 50~100으로 늘려보세요)*

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "random_state": SEED,
        "n_jobs": -1,
        "verbose": -1
    }

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    # Optuna는 최소화 방향으로 최적화하므로 RMSLE 반환
    return rmsle(y_val, pred)

optuna.logging.set_verbosity(optuna.logging.WARNING) # 로그 깔끔하게 정리
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print(f"Best Optuna Params: {study.best_params}")

# 최적 파라미터로 모델 재학습
best_lgbm = LGBMRegressor(**study.best_params, random_state=SEED, n_jobs=-1, verbose=-1)
best_lgbm.fit(X_train, y_train)
tuned_pred = best_lgbm.predict(X_val)
tuned_result = evaluate_regr(y_val, tuned_pred, name="Tuned LightGBM")

## 앙상블 (Voting Regressor)
단일 모델의 과적합을 방지하고 예측 안정성을 높이기 위해, 성능이 좋은 XGBoost와 튜닝된 LightGBM을 결합합니다.

In [ ]:
# XGBoost 모델 불러오기
xgb_model = models["XGBoost"]

# 두 모델의 예측 결과를 평균 내는 VotingRegressor
voting_model = VotingRegressor(
    estimators=[
        ("XGB", xgb_model),
        ("LGBM_Tuned", best_lgbm)
    ]
)

voting_model.fit(X_train, y_train)
voting_pred = voting_model.predict(X_val)
voting_result = evaluate_regr(y_val, voting_pred, name="Voting Ensemble (XGB + Tuned LGBM)")

## 최종 모델 선택
Validation 데이터에서 성능이 가장 좋았던 앙상블 모델을 최종 배포용 모델(`final_model`)로 선정합니다.

In [ ]:
# Validation 결과 비교 후 최종 모델 선택 (이 코드에서는 앙상블 모델로 지정)
final_model = voting_model
print("\n최종 모델이 선택되었습니다:", type(final_model).__name__)

## Test셋 최종 평가
숨겨두었던 Test 데이터로 딱 한 번 최종 성능을 측정합니다.

In [ ]:
test_pred = final_model.predict(X_test)
final_result = evaluate_regr(y_test, test_pred, name="최종 모델 (Test셋)")

print("\nBaseline(Validation) 대비 개선 내역:")
print(f"  RMSLE: {baseline_result['rmsle']:.4f} -> {final_result['rmsle']:.4f}")

## 모델 저장 (pkl) & 저장된 모델 검증
학습된 최종 모델을 파일로 저장하고, 제대로 불러와서 API 서버에서 예측할 수 있는지 가상의 데이터를 넣어 검증합니다.

In [ ]:
# pkl 저장
output_dir = Path("..") / "models_pkl"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "hangang_parking.pkl" # 파일명은 프로젝트에 맞게 수정 가능
joblib.dump(final_model, model_path)

size_kb = model_path.stat().st_size / 1024
print(f"모델 저장 완료: {model_path.resolve()} ({size_kb:.1f} KB)")

# 저장된 모델 검증
loaded_model = joblib.load(model_path)
print(f"\n모델 로드 성공: {type(loaded_model).__name__}")

# 가상의 테스트 데이터 입력 및 결과 확인
test_cases = [
    ([1, 458, 8, 7, 1, 33, 1], "스테이션A 8월 일요일(공휴일)"),
    ([2, 532, 1, 2, 0, 2,  0], "스테이션B 1월 월요일"),
    ([6, 610, 5, 6, 1, 20, 0], "스테이션C 5월 토요일"),
]

print("\n[예측 테스트]")
for case, label in test_cases:
    X_input = pd.DataFrame([case], columns=FEATURE_COLUMNS)
    pred = float(loaded_model.predict(X_input)[0])
    pred = max(0.0, min(100.0, pred)) # 결과값을 0~100 사이로 보정
    print(f"  {label}: 수요도 {pred:.1f}%")